# Sub-Aperture Decomposition

**Source script:** `grdl/example/image_processing/sar/sublook_compare.py`

This notebook demonstrates how to split a focused SAR image (SICD) into azimuth
sub-aperture "looks". Sub-aperture decomposition is the foundational technique for:

- **Coherent Change Detection (CCD)** — comparing sub-look coherence across time
- **Dominance-based detection** — identifying scatterers that concentrate energy in few looks
- **CSI (Coherent Shape Index)** — encoding scattering mechanism from sub-look phase
- **Moving target indication** — targets that shift position across looks

## Concept

A SICD image is formed from the full synthetic aperture. The azimuth frequency axis
encodes Doppler (angular position in the aperture). Partitioning this axis into N
non-overlapping sub-bands produces N independent "looks" — each formed from a fraction
of the aperture, viewing the scene from a slightly different angle.

## Data Setup

Set `SICD_PATH` in Cell 3 to point to any SICD NITF file.

**Default test path:** `/data/sar/from_duane/SICD/` — Umbra spotlight collects.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")

In [ ]:
from pathlib import Path
import numpy as np

from grdl.IO import SICDReader
from grdl.data_prep import ChipExtractor
from grdl.image_processing.sar import SublookDecomposition
from grdl.image_processing.intensity import ToDecibels, PercentileStretch

# ════════════════════════════════════════════════════════════════════
# USER CONFIGURATION
# ════════════════════════════════════════════════════════════════════
SICD_PATH = Path("/data/sar/from_duane/SICD/2023-01-23-18-25-33_UMBRA-05_SICD.nitf")
CHIP_SIZE = 2048      # pixels per side for center chip
NUM_LOOKS = 3         # number of azimuth sub-apertures
OVERLAP = 0.0         # spectral overlap between sub-bands (0.0 = independent)
PLOW = 2.0            # lower percentile for display stretch
PHIGH = 98.0          # upper percentile for display stretch

assert SICD_PATH.exists(), f"SICD file not found: {SICD_PATH}"
print(f"Target: {SICD_PATH.name}")

## 1. Open SICD & Inspect Metadata

The `SICDReader` provides:
- `metadata` — full SICDMetadata (collection geometry, grid, timeline)
- `get_shape()` → `(rows, cols)` of the full image
- `read_chip(r0, r1, c0, c1)` → complex64 array for a sub-region

In [ ]:
with SICDReader(SICD_PATH) as reader:
    meta = reader.metadata
    rows, cols = reader.get_shape()
    print(f"Image size       : {rows} x {cols}")
    print(f"Collection time  : {meta.timeline.collect_start}")
    print(f"Duration         : {meta.timeline.collect_duration:.4f} s")

    ci = meta.collection_info
    if ci:
        print(f"Collector        : {ci.collector_name}")

    # Plan center chip (index-only — no data read yet)
    extractor = ChipExtractor(nrows=rows, ncols=cols)
    region = extractor.chip_at_point(
        rows // 2, cols // 2,
        row_width=CHIP_SIZE, col_width=CHIP_SIZE,
    )
    chip_h = region.row_end - region.row_start
    chip_w = region.col_end - region.col_start
    print(f"\nCenter chip      : [{region.row_start}:{region.row_end}, "
          f"{region.col_start}:{region.col_end}]")
    print(f"Chip dimensions  : {chip_h} x {chip_w}")

    # Read complex pixels
    chip = reader.read_chip(
        region.row_start, region.row_end,
        region.col_start, region.col_end,
    )

print(f"\nChip shape: {chip.shape}, dtype: {chip.dtype}")
print(f"Memory: {chip.nbytes / 1e6:.1f} MB")

## 2. Sub-Aperture Decomposition

`SublookDecomposition` partitions the azimuth frequency axis into `num_looks`
sub-bands, each windowed and IFFT'd back to the spatial domain.

The `metadata` parameter is required — it carries the grid parameters
(row/col spacing, frequency support) needed to compute the filter bank.

Output shape: `(num_looks, rows, cols)` complex64.

In [ ]:
sublook = SublookDecomposition(
    meta,
    num_looks=NUM_LOOKS,
    dimension='azimuth',
    overlap=OVERLAP,
)

looks = sublook.decompose(chip)
print(f"Sublook stack shape: {looks.shape}, dtype: {looks.dtype}")
print(f"Memory: {looks.nbytes / 1e6:.1f} MB")

## 3. Display Preparation

Convert complex sub-looks to display-ready imagery:
1. `ToDecibels` — `|z|²` → dB (20·log10|z|)
2. `PercentileStretch` — map `[p_low, p_high]` → `[0, 1]`

These are composable GRDL processors that operate on the full stack at once.

In [ ]:
to_db = ToDecibels()
stretch = PercentileStretch(plow=PLOW, phigh=PHIGH)

# Full aperture chip → display
chip_display = stretch.apply(to_db.apply(chip))
print(f"Full aperture display: {chip_display.shape}, range [{chip_display.min():.3f}, {chip_display.max():.3f}]")

# Sub-looks → display
looks_display = stretch.apply(to_db.apply(looks))
print(f"Sublook display: {looks_display.shape}, range [{looks_display.min():.3f}, {looks_display.max():.3f}]")

## 4. Visualization with napari

View the full-aperture image and sub-looks as separate layers.
Use the layer visibility toggles to compare fore/mid/aft looks.

**Interpretation:**
- Targets present in all 3 looks → stable scatterers (buildings, poles)
- Targets flickering between looks → motion, layover, or vegetation

In [ ]:
import napari

viewer = napari.Viewer(title="Sublook Decomposition")

# Full aperture as base layer
viewer.add_image(
    chip_display,
    name="Full Aperture",
    colormap="gray",
    contrast_limits=[0, 1],
)

# Sub-looks as overlay layers
look_names = ["Fore (look 1)", "Mid (look 2)", "Aft (look 3)"] if NUM_LOOKS == 3 else None
for i in range(NUM_LOOKS):
    name = look_names[i] if look_names else f"Sub-look {i+1}"
    viewer.add_image(
        looks_display[i],
        name=name,
        colormap="gray",
        contrast_limits=[0, 1],
        visible=(i == 0),  # only first look visible by default
    )

print(f"napari viewer opened with {NUM_LOOKS + 1} layers")
print("Toggle layer visibility to compare sub-aperture looks.")

## 5. Quantitative Comparison

Verify energy conservation: the sum of sub-look powers should
approximate the full aperture power (within numerical precision).

In [ ]:
# Energy conservation check
full_power = np.sum(np.abs(chip) ** 2)
sublook_power = np.sum(np.abs(looks) ** 2)

ratio = sublook_power / full_power
print(f"Full aperture power : {full_power:.6e}")
print(f"Sum of look powers  : {sublook_power:.6e}")
print(f"Ratio               : {ratio:.6f}")
print(f"{'✓ Energy conserved' if abs(ratio - 1.0) < 0.05 else '⚠ Energy mismatch — check overlap setting'}")

# Per-look power distribution
for i in range(NUM_LOOKS):
    lp = np.sum(np.abs(looks[i]) ** 2)
    print(f"  Look {i+1} power fraction: {lp/full_power:.4f}")

---

## Summary

This notebook replaces `grdl/example/image_processing/sar/sublook_compare.py`.

Key patterns:
- `ChipExtractor` for index-only region planning (no data copy)
- `SublookDecomposition(meta, ...)` requires SICD metadata for grid parameters
- `with SICDReader(...) as reader:` ensures NITF handles are released
- `ToDecibels` + `PercentileStretch` are composable display processors
- napari provides GPU-accelerated tiled rendering for large arrays